In [ ]:
# source: https://youtu.be/fxx_E0ojKrc 

In [ ]:
# conda install -c conda-forge statsforecast utilsforecast

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from utilsforecast.plotting import plot_series # type: ignore
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import *


## EDA

In [ ]:
df = pd.read_csv('dataset/daily_sales_french_bakery.csv', parse_dates=['ds'])
df.info()

In [ ]:
df.sample(5)

In [ ]:
df['unique_id'].value_counts()

In [ ]:
df = df.groupby('unique_id').filter(lambda x : len(x) >= 28)
df = df.drop('unit_price', axis=1)
df.sample(5)

In [ ]:
df['unique_id'].value_counts()

In [ ]:
plot_series(df=df, ids=['BAGUETTE', 'CROISSANT'], palette='viridis')

In [ ]:

plot_series(df=df, ids=['BAGUETTE', 'CROISSANT'], palette='viridis', max_insample_length=56)

## Baseline Model Training

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import Naive, HistoricAverage, WindowAverage, SeasonalNaive

In [ ]:
horizon = 7
models = [
    Naive(),
    HistoricAverage(),
    WindowAverage(window_size=7),
    SeasonalNaive(season_length=7)
]

sf = StatsForecast(models=models, freq='D')
sf.fit(df=df)

preds = sf.predict(h=horizon)

In [ ]:
preds.head(20)

In [ ]:
preds = preds.reset_index()
preds

In [ ]:
plot_series(df=df, ids=['BAGUETTE', 'CROISSANT'], forecasts_df=preds, max_insample_length=28, palette='viridis')

## Baseline Model Evaluation

In [ ]:
test = df.groupby('unique_id').tail(7)
train = df.drop(test.index).reset_index(drop=True)

sf.fit(df=train)
preds = sf.predict(h=horizon)

In [ ]:

eval_df = pd.merge(test, preds, 'left', ['ds', 'unique_id'])
evaluation = evaluate(df=eval_df, metrics=[mae])
evaluation.head()

In [ ]:
evaluation = evaluation.drop('unique_id', axis=1).groupby('metric').mean().reset_index()
evaluation

In [ ]:
methods = evaluation.columns[1:].tolist()
values = evaluation.iloc[0, 1:].tolist()

plt.figure(figsize=(10,6))
bars = plt.bar(methods, values)
print(bars[0])
for bar, value in zip(bars, values):
    plt.text(
        x=bar.get_x() + bar.get_width()/2, 
        y=bar.get_height() + 0.05,
        s=f'{value:.3f}',
        ha='center',
        va='bottom',
        fontweight='bold'
    )

plt.xlabel('Methods')
plt.ylabel('MAE')
plt.tight_layout()
# plt.show()

## AutoARIMA    

In [ ]:
from statsforecast.models import AutoARIMA

In [ ]:
unique_ids = ['BAGUETTE', 'CROISSANT']

small_train = train[train['unique_id'].isin(unique_ids)]
small_test = test[test['unique_id'].isin(unique_ids)]

models = [
    AutoARIMA(seasonal=False, alias='ARIMA'),
    AutoARIMA(season_length=7, alias='SARIMA')
]

sf = StatsForecast(models=models, freq='D')
sf.fit(df=small_train)
arima_preds = sf.predict(h=horizon)